# NUMPY MASTERY — CHAPTERS 10–11

```
NUMPY MASTERY
Current Chapters: 10 (Axis Mastery), 11 (Broadcasting Mastery)
Progress:         11 / 75 chapters
Completed:        Ch1-9
Remaining:        64 chapters
```

These are, by a wide margin, the two most important conceptual chapters in this entire course.
Nearly every confusing bug in pandas, scikit-learn, PyTorch, and TensorFlow code traces back to
a shaky mental model of **axis** and **broadcasting**. Take this notebook slowly.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 10 — Axis Mastery

## Concept — the reasoning tool, not the memorized rule

Forget "axis=0 is rows, axis=1 is columns." That phrase only works for 2D and actively misleads
you in 3D+. Use this reasoning instead:

> **`axis=k` names the dimension that gets COLLAPSED (summarized away) by the operation.**
> Every other dimension survives, in order, in the result.

So `arr.sum(axis=0)` means: "walk along axis 0, collapsing it into one number per position of
all the *other* axes." The result's shape is `arr.shape` with axis 0 **removed**.

## Why it matters
Every reduction in ML — averaging over samples, summing over features, reducing over channels,
pooling over spatial dimensions — is an axis operation. Getting the axis wrong doesn't crash;
it silently computes the wrong statistic, which is far more dangerous.

## The universal rule, formalized
```
result_shape = tuple(dim for i, dim in enumerate(arr.shape) if i != axis)   # (keepdims=False)
```


In [ ]:
arr2d = np.arange(1, 7).reshape(2, 3)
print("arr2d shape:", arr2d.shape)
print(arr2d)
print()
print("arr2d.sum(axis=0) :", arr2d.sum(axis=0), " shape:", arr2d.sum(axis=0).shape)
print("arr2d.sum(axis=1) :", arr2d.sum(axis=1), " shape:", arr2d.sum(axis=1).shape)


### Expected Output
```
arr2d shape: (2, 3)
[[1 2 3]
 [4 5 6]]

arr2d.sum(axis=0) : [5 7 9]       shape: (3,)
arr2d.sum(axis=1) : [ 6 15]       shape: (2,)
```

### Reasoning it through (don't memorize — derive)
- `arr2d.shape = (2, 3)` → axis 0 has size 2, axis 1 has size 3.
- `sum(axis=0)` collapses axis 0 (the size-2 dimension) → remaining shape is `(3,)` → you get
  one number **per column**, each the sum **down** that column (across the 2 rows).
- `sum(axis=1)` collapses axis 1 (the size-3 dimension) → remaining shape is `(2,)` → you get
  one number **per row**, each the sum **across** that row (across the 3 columns).

**Mental shortcut:** "collapsing axis k" = "summing over every position along axis k, for each
fixed combination of the other axes."


In [ ]:
# --- Now the 3D case, where "rows/columns" language breaks down ---
arr3d = np.arange(24).reshape(2, 3, 4)   # think: (batch=2, height=3, width=4)
print("arr3d.shape:", arr3d.shape)

s0 = arr3d.sum(axis=0)
s1 = arr3d.sum(axis=1)
s2 = arr3d.sum(axis=2)

print("sum(axis=0).shape :", s0.shape, " <- collapsed 'batch', left with (height, width)")
print("sum(axis=1).shape :", s1.shape, " <- collapsed 'height', left with (batch, width)")
print("sum(axis=2).shape :", s2.shape, " <- collapsed 'width', left with (batch, height)")


### Expected Output
```
arr3d.shape: (2, 3, 4)
sum(axis=0).shape : (3, 4)  <- collapsed 'batch', left with (height, width)
sum(axis=1).shape : (2, 4)  <- collapsed 'height', left with (batch, width)
sum(axis=2).shape : (2, 3)  <- collapsed 'width', left with (batch, height)
```

This is exactly the reasoning you'll use for:
- Averaging a batch of images over the batch dimension → `axis=0`
- Averaging RGB channels of a single image → `axis=-1` (or `axis=2`)
- Computing per-feature mean over samples in a `(n_samples, n_features)` matrix → `axis=0`
  (this is *the* single most common axis choice in ML preprocessing — collapse samples, keep
  one statistic per feature)

### `keepdims=True` — the parameter that prevents a huge class of broadcasting bugs


In [ ]:
X = np.arange(1, 7).reshape(2, 3).astype(float)
print("X:\n", X)

mean_no_keepdims = X.mean(axis=1)
mean_keepdims    = X.mean(axis=1, keepdims=True)

print("mean(axis=1)                  :", mean_no_keepdims, " shape:", mean_no_keepdims.shape)
print("mean(axis=1, keepdims=True)   :\n", mean_keepdims, " shape:", mean_keepdims.shape)

# WHY keepdims matters: subtracting the per-row mean back from X
print()
print("X - mean_keepdims (correct, broadcasts row-wise):\n", X - mean_keepdims)


### Expected Output
```
X:
 [[1. 2. 3.]
  [4. 5. 6.]]
mean(axis=1)                  : [2. 5.]  shape: (2,)
mean(axis=1, keepdims=True)   :
 [[2.]
  [5.]]  shape: (2, 1)

X - mean_keepdims (correct, broadcasts row-wise):
 [[-1.  0.  1.]
  [-1.  0.  1.]]
```

Without `keepdims=True`, `mean_no_keepdims` has shape `(2,)`, which — as you'll see in
Chapter 11 — broadcasts against `X`'s shape `(2,3)` in the **wrong** way (it tries to align
against the *last* axis, i.e. columns, not rows), producing silently wrong results or a shape
error. `keepdims=True` preserves the collapsed axis as size 1, so the subsequent broadcast
against the original array aligns correctly. **This is one of the most common real-world NumPy
bugs** — always reach for `keepdims=True` when you plan to broadcast a reduction back against
the original array.

### Common Mistake
Assuming axis behavior is symmetric between `.sum()`/`.mean()` and functions like
`np.concatenate()`/`np.stack()` — for those, `axis` means "where to join," a related but
different meaning we'll cover in Chapter 18. Always check the specific function's docstring
mental model, but the "what gets collapsed / what's being indexed" instinct transfers.

### Interview Insight
> "Given a `(32, 128, 128, 3)` batch of images, how would you compute the per-channel mean
> across the whole batch?"
Collapse batch, height, and width — keep the channel axis: `images.mean(axis=(0,1,2))` →
shape `(3,)`. Multi-axis reduction (a tuple for `axis`) is exactly this "collapse several
axes, keep the rest" reasoning generalized.


---
# CHAPTER 11 — Broadcasting Mastery

## Concept
Broadcasting lets NumPy perform element-wise operations on arrays of **different shapes**
without actually copying data to make them the same shape — it's a set of rules for when
shapes are "compatible enough" to align virtually.

## The Broadcasting Rule (memorize this precisely — it's always applied right-to-left)

To check if two shapes are broadcast-compatible, align them **from the right (trailing
dimensions)**, and for each pair of aligned dimensions:
1. If the dimensions are **equal**, they're compatible.
2. If one of them is **1**, it gets "stretched" (conceptually) to match the other.
3. If a shape has **fewer dimensions**, it's padded with size-1 dimensions on the **left**
   (the missing/leading axes).
4. If neither of the above holds (unequal, neither is 1), it's an error.

```
Shape A: (3, 4)
Shape B:    (4,)
                       align right:
A:  3  4
B:     4        <- B is padded to (1, 4) conceptually, then dim 4==4 matches, dim 1 stretches to 3
Result: (3, 4)                                     -- valid --
```


In [ ]:
# --- Case 1: array + scalar (broadcasting a 0-d shape against anything) ---
a = np.array([[1, 2, 3], [4, 5, 6]])
print("a + 10 :\n", a + 10)

# --- Case 2: matrix (3,4) + vector (4,) -- vector broadcasts across ROWS ---
matrix = np.arange(12).reshape(3, 4)
vector = np.array([100, 200, 300, 400])
print("\nmatrix.shape:", matrix.shape, " vector.shape:", vector.shape)
print("matrix + vector:\n", matrix + vector)

# --- Case 3: matrix (3,4) + column vector (3,1) -- broadcasts across COLUMNS ---
col = np.array([[1], [2], [3]])
print("\nmatrix.shape:", matrix.shape, " col.shape:", col.shape)
print("matrix + col:\n", matrix + col)


### Expected Output
```
a + 10 :
 [[11 12 13]
  [14 15 16]]

matrix.shape: (3, 4)  vector.shape: (4,)
matrix + vector:
 [[100 201 302 403]
  [104 205 306 407]
  [108 209 310 411]]

matrix.shape: (3, 4)  col.shape: (3, 1)
matrix + col:
 [[ 1  2  3  4]
  [ 6  7  8  9]
  [11 12 13 14]]
```

### Walking through Case 2 by the rule
```
matrix: (3, 4)
vector:    (4,)   -> padded to (1, 4)
compare right-to-left:
  dim -1: 4 vs 4  -> equal, OK
  dim -2: 3 vs 1  -> one of them is 1, stretch the "1" to 3, OK
result shape: (3, 4)
```
The `(4,)` vector is treated as **one row**, conceptually repeated for each of the 3 rows of
`matrix` — this is exactly "add this same row to every row."

### Walking through Case 3
```
matrix: (3, 4)
col:    (3, 1)
compare right-to-left:
  dim -1: 4 vs 1  -> one of them is 1, stretch the "1" to 4, OK
  dim -2: 3 vs 3  -> equal, OK
result shape: (3, 4)
```
The `(3, 1)` column is treated as **one value per row**, conceptually repeated across all 4
columns — "add this per-row value to every column of that row." This is *exactly* the pattern
from Chapter 10's `keepdims=True` example.

### Common broadcasting errors — learn to read the message


In [ ]:
try:
    bad = np.arange(12).reshape(3, 4) + np.array([1, 2, 3])   # (3,4) vs (3,) -- INCOMPATIBLE
except ValueError as e:
    print("ValueError:", e)


### Expected Output
```
ValueError: operands could not be broadcast together with shapes (3,4) (3,)
```

### Diagnosing this error
```
matrix: (3, 4)
other:     (3,)  -> padded to (1, 3)
compare right-to-left:
  dim -1: 4 vs 3  -> NOT equal, NEITHER is 1  ->  INCOMPATIBLE, error
```
The fix depends on intent: if you meant "add a value per row," you needed shape `(3, 1)`
(`.reshape(3, 1)` or `[:, np.newaxis]`), not `(3,)`. If you meant "add a value per column," you
needed shape `(4,)`, not `(3,)`. **The error message tells you the misaligned shapes — always
work backward from it using the right-to-left rule, don't guess-and-check.**


In [ ]:
# --- Harder cases: 3D broadcasting ---
batch = np.ones((2, 3, 4))          # e.g., 2 samples, 3x4 "image"
row_bias   = np.arange(4)           # (4,)      -> broadcasts against the LAST axis
col_bias   = np.arange(3).reshape(3, 1)   # (3,1) -> broadcasts against the middle axis

print("batch.shape                :", batch.shape)
print("(batch + row_bias).shape   :", (batch + row_bias).shape)
print("(batch + col_bias).shape   :", (batch + col_bias).shape)

# The classic outer-product-via-broadcasting trick:
x = np.array([1, 2, 3])          # shape (3,)
y = np.array([10, 20])           # shape (2,)
outer = x[:, np.newaxis] * y[np.newaxis, :]   # (3,1) * (1,2) -> (3,2)
print("\nouter product via broadcasting:\n", outer, " shape:", outer.shape)


### Expected Output
```
batch.shape                : (2, 3, 4)
(batch + row_bias).shape   : (2, 3, 4)
(batch + col_bias).shape   : (2, 3, 4)

outer product via broadcasting:
 [[10 20]
  [20 40]
  [30 60]]  shape: (3, 2)
```

### The outer-product trick, explained
```
x[:, np.newaxis]  shape: (3, 1)
y[np.newaxis, :]  shape: (1, 2)
right-to-left:
  dim -1: 1 vs 2 -> stretch 1 to 2
  dim -2: 3 vs 1 -> stretch 1 to 3
result: (3, 2)  -- every combination of x[i] * y[j], computed with NO explicit loop
```
This exact pattern — reshape to insert size-1 axes, let broadcasting do an implicit "outer"
combination — powers pairwise distance calculations (Chapter 41, KNN), attention-like
score matrices, and much of vectorized simulation (Chapter 21, 68).

### Why broadcasting can still blow up your memory
Broadcasting avoids *storing* the stretched copies during the description of the operation, but
the **output** of the operation is still a full, fully-materialized array at the broadcast
shape. `(1_000_000, 1) + (1, 1_000_000)` broadcasts to `(1_000_000, 1_000_000)` — that's 10^12
elements, ~8 TB in float64. Broadcasting doesn't make an operation "free" — it just avoids
redundant *input* storage; the *output* size is exactly what you'd expect from the result
shape, and can be huge. This is a favorite "gotcha" interview/production question (Chapter 71,
Scenario 3).

### Common Mistakes
- Forgetting to insert `np.newaxis` / use `keepdims=True` and getting a broadcast against the
  wrong axis (silently wrong, not an error, if shapes happen to be compatible in the wrong way).
- Assuming broadcasting always "just works" without checking the resulting shape — always
  predict, then verify with `.shape`.

### Interview Insight
> "Explain broadcasting rules and give an example of a dangerous broadcast."
State the right-to-left, pad-with-1, stretch-1 rule precisely. For "dangerous": two arrays with
shapes `(N, 1)` and `(1, N)` broadcast into an `(N, N)` array — completely valid, no error, but
can silently exhaust memory for large `N`. This shows you understand broadcasting isn't just
"does it work," but "what does it cost."


---
## Chapters 10–11 Summary

- **Axis** = the dimension that gets **collapsed**; every other dimension survives in order.
  Don't memorize "0=rows, 1=columns" — derive the result shape by removing the axis size from
  the shape tuple. This generalizes cleanly to 3D/4D where row/column language breaks down.
- **`keepdims=True`** preserves the collapsed axis as size 1 — essential whenever you plan to
  broadcast a reduction back against the original array (e.g., subtracting a per-row mean).
- **Broadcasting rule**: align shapes right-to-left; dimensions are compatible if equal or one
  is 1; missing leading dimensions are padded with 1. This single rule explains scalar
  broadcasting, row/column broadcasting, and ND broadcasting uniformly.
- Broadcasting avoids storing redundant **input** copies but the **output** is always fully
  materialized at the broadcast shape — this can silently create huge arrays and exhaust memory
  even though "no error occurred."

---

## Mastery Exercises

**Exercise 1 (Prediction — do this before running anything):** For `X` of shape `(500, 20)`
(500 samples, 20 features), predict the shape of each of these, and whether it's even valid:
- `X.mean(axis=0)`
- `X.mean(axis=1)`
- `X - X.mean(axis=0)`
- `X - X.mean(axis=1)`
- `X - X.mean(axis=1, keepdims=True)`

**Exercise 2 (Broadcasting compatibility):** For each shape pair, decide if broadcasting
succeeds, and if so, the result shape. If not, identify exactly which trailing dimension pair
fails and why.
- `(8, 1, 6, 1)` and `(7, 1, 5)`
- `(3, 5)` and `(5,)`
- `(3, 5)` and `(3,)`
- `(2, 1)` and `(8, 4, 3)`

**Coding Challenge:** Implement `standardize(X)` that z-score normalizes a `(n_samples,
n_features)` matrix **per feature** (subtract per-feature mean, divide by per-feature std),
using broadcasting correctly with `keepdims=True`, and add an assertion that guards against
dividing by zero std (a constant feature column).

**Debugging Challenge:**
```python
X = np.random.default_rng(0).normal(size=(100, 5))
row_means = X.mean(axis=1)          # shape (100,)
centered = X - row_means            # this either errors OR silently does the wrong thing
```
Determine: does this raise an error or silently misbehave? Explain exactly why using the
broadcasting rule, and give the one-line fix.

**Interview Questions:**
1. State the broadcasting rule precisely (not "NumPy figures it out").
2. Why is `keepdims=True` important when you intend to re-broadcast a reduction?
3. Give an example of a broadcasting operation that is valid but dangerous from a memory
   perspective.
4. For a `(32, 128, 128, 3)` image batch, which axis (or axes) would you collapse to compute
   the mean pixel value per channel across the whole batch?

**Quiz:**
- Q1: `arr.shape == (5, 1, 3)`. After `arr.sum(axis=1)`, what is the resulting shape?
- Q2: True/False — Broadcasting always avoids allocating a full-size output array.
- Q3: Shapes `(4,)` and `(3, 1)` — do they broadcast, and if so, to what shape?

---

